<a href="https://colab.research.google.com/github/ShikharVeer10/TokenCompression/blob/main/Token_Compression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **PHASE 1**

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ GPU Available: {gpu_name}")
    print(f"✅ Total VRAM: {total_memory:.2f} GB")
else:
    print("❌ No GPU found! Please change your runtime to T4 GPU.")

✅ GPU Available: Tesla T4
✅ Total VRAM: 14.56 GB


In [ ]:
!pip install --upgrade pip setuptools wheel
!pip install -q -U "tokenizers>=0.19.1"
!pip install -q -U transformers bitsandbytes accelerate pydantic psutil

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
import transformers
import bitsandbytes
import accelerate

print("Transformers:", transformers.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("Accelerate:", accelerate.__version__)

Transformers: 5.16.1
BitsAndBytes: 0.50.2
Accelerate: 1.14.0


In [ ]:
import torch
from transformers import BitsAndBytesConfig

# Define the 4-bit configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",               # High-precision 4-bit format
    bnb_4bit_compute_dtype=torch.float16,    # Math happens in float16
    bnb_4bit_use_double_quant=True           # Extra memory compression
)

print("BitsAndBytes 4-bit configuration is ready.")

BitsAndBytes 4-bit configuration is ready.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-7B-Instruct"

print("Downloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Downloading and loading 4-bit base model (takes ~2-4 mins)...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager"  # Vital for later attention hooks
)

print("Model successfully loaded into GPU memory!")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model successfully loaded into GPU memory!


In [ ]:
import time
import torch

def get_vram_usage():
    """Returns allocated and peak VRAM in Gigabytes."""
    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    peak = torch.cuda.max_memory_allocated() / (1024 ** 3)
    return allocated, peak

def run_baseline_inference(prompt: str, max_new_tokens: int = 128):
    # Reset GPU peak tracking so we measure only this run
    torch.cuda.reset_peak_memory_stats()

    # Tokenize input text
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_token_count = inputs.input_ids.shape[1]

    # Start timer
    start_time = time.time()

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # Deterministic for exact benchmarking
            use_cache=True     # Standard full uncompressed KV-cache
        )

    latency = time.time() - start_time
    output_token_count = outputs.shape[1] - input_token_count
    allocated_vram, peak_vram = get_vram_usage()

    # Decode model output
    response_text = tokenizer.decode(outputs[0][input_token_count:], skip_special_tokens=True)

    return {
        "response": response_text,
        "input_tokens": input_token_count,
        "output_tokens": output_token_count,
        "latency_sec": round(latency, 2),
        "tokens_per_sec": round(output_token_count / latency, 2),
        "peak_vram_gb": round(peak_vram, 2)
    }

print("Baseline testing function created successfully.")

Baseline testing function created successfully.


In [ ]:
test_prompt = """<|im_start|>system
You are a concise technical assistant.<|im_end|>
<|im_start|>user
Explain in two sentences what a Key-Value Cache is in Large Language Models.<|im_end|>
<|im_start|>assistant
"""

baseline_results = run_baseline_inference(test_prompt, max_new_tokens=100)

print("--- BASELINE UNCOMPRESSED METRICS ---")
print(f"Prompt Tokens      : {baseline_results['input_tokens']}")
print(f"Generated Tokens   : {baseline_results['output_tokens']}")
print(f"Total Latency      : {baseline_results['latency_sec']} seconds")
print(f"Throughput         : {baseline_results['tokens_per_sec']} tokens/sec")
print(f"Peak VRAM Footprint: {baseline_results['peak_vram_gb']} GB")
print("-" * 36)
print(f"Model Output:\n{baseline_results['response']}")

--- BASELINE UNCOMPRESSED METRICS ---
Prompt Tokens      : 37
Generated Tokens   : 49
Total Latency      : 8.4 seconds
Throughput         : 5.83 tokens/sec
Peak VRAM Footprint: 5.19 GB
------------------------------------
Model Output:
A Key-Value Cache in Large Language Models stores recently accessed tokens and their corresponding embeddings to quickly retrieve them, reducing computational load during inference. This caching mechanism significantly speeds up the model's response time by avoiding redundant computations for frequently accessed data.




---

## **PHASE 2**

In [ ]:
!pip install -q llmlingua

In [ ]:
from llmlingua import PromptCompressor

print("Loading LLMLingua-2 model onto GPU...")

compressor = PromptCompressor(
    model_name="microsoft/llmlingua-2-xlm-roberta-large-meetingbank",
    use_llmlingua2=True,
    device_map="cuda"
)

print("LLMLingua-2 loaded successfully!")

Loading LLMLingua-2 model onto GPU...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

LLMLingua-2 loaded successfully!


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

# 1. Strict Data Contract using Pydantic
class CompressionRequest(BaseModel):
    context: str
    instruction: Optional[str] = Field(default="", description="The query or task instruction")
    rate: float = Field(default=0.5, ge=0.1, le=0.9, description="Target retention rate (0.5 means keep 50% of tokens)")

class CompressionResponse(BaseModel):
    compressed_text: str
    origin_tokens: int
    compressed_tokens: int
    tokens_saved: int
    pruning_percentage: float

# 2. Wrapper function using LLMLingua-2
def compress_context(request: CompressionRequest) -> CompressionResponse:
    # Essential structural punctuation marks to preserve
    protected_tokens = ['\n', '?', '.', ':', '-', '"', "'"]

    results = compressor.compress_prompt(
        context=[request.context],
        instruction=request.instruction,
        rate=request.rate,
        force_tokens=protected_tokens
    )

    orig = results['origin_tokens']
    comp = results['compressed_tokens']
    saved = orig - comp
    savings_pct = round((saved / orig) * 100, 2) if orig > 0 else 0.0

    return CompressionResponse(
        compressed_text=results['compressed_prompt'],
        origin_tokens=orig,
        compressed_tokens=comp,
        tokens_saved=saved,
        pruning_percentage=savings_pct
    )

print("Compression wrapper created successfully.")

Compression wrapper created successfully.


In [ ]:
# A realistic wordy technical scenario
sample_context = """
Dear infrastructure engineering team, I am writing this incident report to formally notify you
that we experienced an unexpected hardware fault in cluster region US-East-4 at approximately
03:45 AM UTC. The secondary cooling distribution unit malfunctioned, causing the server rack
temperatures to surge rapidly from a normal baseline of 42 degrees Celsius up to a critical threshold
of 89 degrees Celsius. Under the high thermal load, node worker-12 experienced memory throttling,
and the distributed cache service threw repeated critical error code 0x884F before shutting down.
Please refer to standard maintenance protocol section 9B to initiate the automated thermal fallback
procedure and redirect network traffic to the secondary standby node immediately.
"""

sample_query = "What was the critical error code, the peak temperature, and which protocol should be followed?"

# Create our structured request targeting a 50% retention rate (rate=0.5)
request = CompressionRequest(
    context=sample_context,
    instruction=sample_query,
    rate=0.5
)

# Run compression
compressed_data = compress_context(request)

print("--- STAGE 1 (LLMLINGUA-2) PRUNING RESULTS ---")
print(f"Original Token Count  : {compressed_data.origin_tokens}")
print(f"Compressed Token Count: {compressed_data.compressed_tokens}")
print(f"Tokens Eliminated     : {compressed_data.tokens_saved}")
print(f"Pruning Percentage    : {compressed_data.pruning_percentage}%")
print("-" * 50)
print("PRUNED TEXT PAYLOAD:")
print(compressed_data.compressed_text)

--- STAGE 1 (LLMLINGUA-2) PRUNING RESULTS ---
Original Token Count  : 143
Compressed Token Count: 79
Tokens Eliminated     : 64
Pruning Percentage    : 44.76%
--------------------------------------------------
PRUNED TEXT PAYLOAD:

 infrastructure engineering team writing incident report
 unexpected hardware fault cluster US-East-4
 03:45 AM UTC. secondary cooling unit malfunctioned server
 temperatures 42 degrees
 89 degrees Celsius. high thermal load node worker-12 memory throttling
 distributed cache error code 0x884F shutting.
 refer maintenance protocol 9B thermal fallback
 redirect network traffic secondary standby node.



In [ ]:
# Format the pruned text into our model's chat template
pruned_prompt = f"""<|im_start|>system
You are a precise technical assistant. Answer the question using only the provided context.<|im_end|>
<|im_start|>user
Context:
{compressed_data.compressed_text}

Question:
{sample_query}<|im_end|>
<|im_start|>assistant
"""

# Run inference with the pruned prompt
pruned_results = run_baseline_inference(pruned_prompt, max_new_tokens=100)

print("--- STAGE 1 INFERENCE RESULTS ---")
print(f"Prompt Tokens Feed to Model: {pruned_results['input_tokens']}")
print(f"Generated Answer Tokens    : {pruned_results['output_tokens']}")
print(f"Execution Latency          : {pruned_results['latency_sec']} seconds")
print(f"Peak VRAM Footprint        : {pruned_results['peak_vram_gb']} GB")
print("-" * 50)
print(f"Answer Generated:\n{pruned_results['response']}")

--- STAGE 1 INFERENCE RESULTS ---
Prompt Tokens Feed to Model: 136
Generated Answer Tokens    : 39
Execution Latency          : 2.83 seconds
Peak VRAM Footprint        : 7.43 GB
--------------------------------------------------
Answer Generated:
The critical error code was 0x884F. The peak temperature was 88 degrees Celsius. The protocol that should be followed is maintenance protocol  tB thermal fallback.


**PHASE-3**

In [ ]:
from pydantic import BaseModel,Field,model_validator

In [ ]:
class SnapKVConfig(BaseModel):
  max_capacity:int=Field(default=512) #Maximum no of tokens to be allowed in memory before deleting
  window_size:int=Field(default=32) #The number of tokens we look to decide what is important
  kernel_size:int=Field(default=5) #This is to cluster attention scores together

  @model_validator(mode="after")
  def validate_logic(self):
    if self.window_size>=self.max_capacity:
      raise ValueError("Window Size should be smaller than the maximum capacity")
    return self

snap_config= SnapKVConfig(max_capacity=512,window_size=32,kernel_size=5)
print("SnapKV rules defined")

In [ ]:
import torch
import torch.nn.functional as F

class SnapKVManager:
  def __init__(self,model,config:SnapKVConfig):
    self.model=model
    self.config=config
    self.hooks=[] #To store hooks to easily remove them later

  def _attention_hook(self,module,inputs,output):
    if len(output)<3 or output[2] is None:
      return output

    attention_weights=output[1]
    past_key_value=output[2] # This represents the actual KV Cache memory
    keys,values=past_key_value[0],past_key_value[1]
    current_memory_size=keys.shape[2]

    if current_memory_size>self.config.max_capacity:
      recent_attention=attention_weights[:,:,:, -self.config.window_size:]
      historical_scores=recent_attention.mean(dim=-2) #To average the attention scores
      smoothed_scores=F.avg_pool1d(historical_scores,kernel_size=self.config.kernel_size,stride=1,padding=self.config.kernel_size//2)
      keep_amount=self.config.max_capacity=self.config.window_size
      _, top_indices=torch.topk(smoothed_scores,k=keep_amount,dim=-1) #Finding the indices for the tokens to stay in original sentence order
      top_indices, _ =torch.sort(top_indices,dim=-1) #Sorting of indices so the tokens stay in original sentence order

      new_keys = torch.cat([torch.gather(keys, 2, top_indices.unsqueeze(-1).expand(-1, -1, -1, keys.shape[-1])),keys[:, :, -self.config.window_size:, :]], dim=2)
      new_values=torch.cat([torch.gather(keys,2,top_indices.unsqueeze(-1).expand(-1,-1,-1,values.shape[-1])),values[:,:, -self.config.window_size:, :]],dim=2)

      return (output[0],output[1], (new_keys,new_values))
    return output

  def turn_on(self):
    for layer in self.model.model.layers:
      hook=layer.self_attn.register_forward_hook(self,_attention_hook)
      self.hooks.append(hook)

  def turn_on(self):
    for hook in self.hooks:
      hook.remove()
    self.hooks=[]



snapkv_engine=SnapKVManager(base_model,snap_config)
print("Initialization of SnapKV Engine")